![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)







### Copyright notice

> <p><small><small>Copyright 2025 DeepMind Technologies Limited.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>

# Manipulation in The Playground! <a href="https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/manipulation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a>

In this notebook, we'll walk through a couple manipulation environments available in MuJoCo Playground.

**A Colab runtime with GPU acceleration is required.** If you're using a CPU-only runtime, you can switch using the menu "Runtime > Change runtime type".


In [1]:
# # #@title Install pre-requisites
!pip install mujoco
!pip install mujoco_mjx
!pip install brax

# !pip install --break-system-packages mediapy -q

In [2]:
!pip install mediapy

In [3]:
# @title Check if MuJoCo installation was successful

import distutils.util
import os
import subprocess

if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.'
  )

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

try:
  print('Checking that the installation succeeded:')
  import mujoco

  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".'
  )

print('Installation successful.')

# Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
xla_flags = os.environ.get('XLA_FLAGS', '')
xla_flags += ' --xla_gpu_triton_gemm_any=True'
os.environ['XLA_FLAGS'] = xla_flags

Tue Apr  7 15:31:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:02:00.0  On |                  Off |
| 30%   31C    P8             18W /  450W |    5365MiB /  24564MiB |     26%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

/tmp/ipykernel_529872/1100621786.py:3: DeprecationWarning: The distutils package is deprecated and slated for removal in Python 3.12. Use setuptools or check PEP 632 for potential alternatives
  import distutils.util


In [4]:
# @title Import packages for plotting and creating graphics
import json
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np

# Graphics and plotting.
# print("Installing mediapy:")
# !command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
# !pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

In [5]:
from jax import config
config.update("jax_enable_x64", True)


In [6]:
# @title Import MuJoCo, MJX, and Brax
from datetime import datetime
import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from brax.training.agents.sac import networks as sac_networks
from brax.training.agents.sac import train as sac
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
from jax import numpy as jp
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp

from brax.training.agents.ars import train as ars

In [7]:
#@title Install MuJoCo Playground
# !pip install playground

In [8]:
#@title Import The Playground

from mujoco_playground import wrapper
from mujoco_playground import registry
from mujoco_playground import registry

In [9]:
print(registry.__file__)

/home/aks-lab/mujoco_playground/mujoco_playground/_src/registry.py


# Manipulation

MuJoCo Playground contains several manipulation environments (all listed below after running the command).

In [10]:
registry.manipulation.ALL_ENVS

('AlohaHandOver',
 'AlohaSinglePegInsertion',
 'PandaPickCube',
 'PandaPickCubeOrientation',
 'PandaPickCubeCartesian',
 'PandaOpenCabinet',
 'PandaRobotiqPushCube',
 'LeapCubeReorient',
 'LeapCubeRotateZAxis',
 'DualUR5eBoxlift')

# Franka Emika Panda

Let's start off with the simplest environment, simply picking up a cube with the Franka Emika Panda.

In [11]:
# env_name = 'PandaPickCubeOrientation'
# env = registry.load(env_name)
# env_cfg = registry.get_default_config(env_name)

In [12]:
env_name = 'DualUR5eBoxlift'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)

INIT NOISE [0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01]
(158,)
(12, 158)

 Default backend: gpu
 Timestep: 0.1 
 CEM Iter: 2 
 Projection Iter: 5 
 Number of batches: 100 
 Number of steps per trajectory: 12 
 Time per trajectory: 1.2000000000000002 
 Number of variables: 144 
 Number of Total constraints: 1080 
 Number of geomteric IDs for colllision: 24


## Train Policy

Let's train the pick cube policy and visualize rollouts. The policy takes roughly 3 minutes to train on an RTX 4090.

In [13]:
CHECKPOINT_IDX = 2
TRAINING = False

In [14]:
from mujoco_playground.config import manipulation_params
ppo_params = manipulation_params.brax_ppo_config(env_name)
ppo_params

action_repeat: 1
batch_size: 256
discounting: 0.97
entropy_cost: 0.01
episode_length: 100
learning_rate: 0.0003
network_factory:
  policy_hidden_layer_sizes: &id001 !!python/tuple
  - 512
  - 256
  - 128
  policy_obs_key: state
  value_hidden_layer_sizes: *id001
  value_obs_key: privileged_state
normalize_observations: true
num_envs: 128
num_evals: 5
num_minibatches: 32
num_resets_per_eval: 1
num_timesteps: 500000
num_updates_per_batch: 2
reward_scaling: 1.0
unroll_length: 10

In [15]:
from mujoco_playground.config import manipulation_params
ppo_params = manipulation_params.brax_ppo_config(env_name)

# # Remove asymmetric observation keys since this env uses flat obs
# if "policy_obs_key" in ppo_params:
#     del ppo_params["policy_obs_key"]
# if "value_obs_key" in ppo_params:
#     del ppo_params["value_obs_key"]

print("Updated ppo_params - removed obs keys")
print(f"policy_obs_key: {ppo_params.get('policy_obs_key')}")
print(f"value_obs_key: {ppo_params.get('value_obs_key')}")

Updated ppo_params - removed obs keys
policy_obs_key: None
value_obs_key: None


In [16]:


# 🔍 DEBUG HERE (before training)
rng = jax.random.PRNGKey(0)
state = env.reset(rng)
data = state.data

print("qpos:", data.qpos.shape, data.qpos.size)
print("qvel:", data.qvel.shape, data.qvel.size)

# target_id = env._mj_model.body_mocapid[
#     env._mj_model.body(name='target_0').id
# ]

# print("target_id:", target_id)
print("mocap_pos:", data.mocap_pos.shape)

print("OBS: state", state.obs['state'].shape)
print("OBS: privileged state", state.obs['privileged_state'].shape)


qpos: (19,) 19
qvel: (18,) 18
mocap_pos: (2, 3)
OBS: state (45,)
OBS: privileged state (52,)


In [17]:


x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory_kwargs = dict(ppo_params.network_factory)
  # Remove asymmetric obs keys — DualUR5eBoxlift returns flat obs, not a dict
  network_factory_kwargs.pop("policy_obs_key", None)
  network_factory_kwargs.pop("value_obs_key", None)
  network_factory = functools.partial(
      ppo_networks.make_ppo_networks,
      **network_factory_kwargs
  )

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    seed=1,
    save_checkpoint_path=os.path.abspath(f"./checkpoints/run_{CHECKPOINT_IDX}"),
)


In [18]:
dict(ppo_training_params)

{'action_repeat': 1,
 'batch_size': 256,
 'discounting': 0.97,
 'entropy_cost': 0.01,
 'episode_length': 100,
 'learning_rate': 0.0003,
 'normalize_observations': True,
 'num_envs': 128,
 'num_evals': 5,
 'num_minibatches': 32,
 'num_resets_per_eval': 1,
 'num_timesteps': 500000,
 'num_updates_per_batch': 2,
 'reward_scaling': 1.0,
 'unroll_length': 10}

### PPO DUAL ARM

In [19]:
print("\n" + "=" * 50)
print(state.obs['state'])
print("\n" + "=" * 50)
print(state.obs['privileged_state'])


[ 1.492 -1.792  1.755 -1.248 -1.598  0.004 -1.494 -1.792  1.742 -1.243 -1.605  0.003 -0.29   0.01
  0.35   1.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.     0.     0.     0.     0.     0.    -0.283 -0.074  0.306  0.995  0.
  0.    -0.096  0.   ]

[ 1.492 -1.792  1.755 -1.248 -1.598  0.004 -1.494 -1.792  1.742 -1.243 -1.605  0.003 -0.29   0.01
  0.35   1.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
  0.     0.     0.     0.     0.     0.     0.     0.     0.    -0.283 -0.074  0.306  0.995  0.
  0.    -0.096  0.     0.     0.     0.     0.     0.     0.     0.   ]


In [20]:
def obs_debug_full(obs_dict, env):
    """
    Exact debugger matching current _get_obs() layout.
    No guessing. No heuristics.
    """

    def debug_single(obs, name):
        idx = 0

        def take(n, label):
            nonlocal idx
            val = obs[idx:idx+n]
            print(f"{label:30s} [{idx}:{idx+n}] → {val}")
            idx += n
            return val

        nv = env._mjx_model.nv
        num_dof = env.num_dof

        print("\n" + "=" * 70)
        print(f"DEBUGGING: {name}")
        print("=" * 70)

        # =========================
        # Core observation
        # =========================

        joint_pos = take(num_dof, "joint_pos")

        ball_pose = take(7, "ball_pose")
        print(f"  ├─ ball_pos  → {ball_pose[:3]}")
        print(f"  └─ ball_quat → {ball_pose[3:]}")

        print("\n--- Velocities ---")
        qvel = take(nv, "qvel")

        joint_vel = qvel[env._joint_mask_vel]

        ball_body_id = env._mj_model.body(name="ball").id
        ball_qvel_idx = env._mj_model.body_dofadr[ball_body_id]

        ball_vel = qvel[ball_qvel_idx:ball_qvel_idx+3]
        ball_ang_vel = qvel[ball_qvel_idx+3:ball_qvel_idx+6]

        print(f"joint_vel     → {joint_vel}")
        print(f"ball_vel      → {ball_vel}")
        print(f"ball_ang_vel  → {ball_ang_vel}")

        print("\n--- Target (world frame) ---")
        target_pos = take(3, "target_pos")
        target_quat = take(4, "target_quat")

        print("\n--- Time ---")
        time = take(1, "time")

        # =========================
        # Privileged tail
        # =========================

        if idx < obs.shape[0]:
            print("\n--- Privileged ---")

            task = take(1, "task")
            success = take(1, "success")
            cost_g_ball = take(1, "cost_g_ball")
            cost_r_ball = take(1, "cost_r_ball")
            current_cost_g = take(1, "current_cost_g")
            current_cost_r = take(1, "current_cost_r")
            penalty_collision = take(1, "penalty_collision_real_time")

        print(f"\nTotal used: {idx} / {obs.shape[0]}")
        print("=" * 70 + "\n")

    # =========================
    # Run for both
    # =========================

    for k, v in obs_dict.items():
        debug_single(v, k)

In [21]:
obs_debug_full(state.obs, env)


DEBUGGING: state
joint_pos                      [0:12] → [ 1.492 -1.792  1.755 -1.248 -1.598  0.004 -1.494 -1.792  1.742 -1.243 -1.605  0.003]
ball_pose                      [12:19] → [-0.29  0.01  0.35  1.    0.    0.    0.  ]
  ├─ ball_pos  → [-0.29  0.01  0.35]
  └─ ball_quat → [1. 0. 0. 0.]

--- Velocities ---
qvel                           [19:37] → [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
joint_vel     → [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
ball_vel      → [0. 0. 0.]
ball_ang_vel  → [0. 0. 0.]

--- Target (world frame) ---
target_pos                     [37:40] → [-0.283 -0.074  0.306]
target_quat                    [40:44] → [ 0.995  0.     0.    -0.096]

--- Time ---
time                           [44:45] → [0.]

Total used: 45 / 45


DEBUGGING: privileged_state
joint_pos                      [0:12] → [ 1.492 -1.792  1.755 -1.248 -1.598  0.004 -1.494 -1.792  1.742 -1.243 -1.605  0.003]
ball_pose                      [12:19] → [-0.29  0.01  0.35  1.    0.    0. 

In [ ]:

if TRAINING:
    make_inference_fn, params, metrics = train_fn(
        environment=env,
        wrap_env_fn=wrapper.wrap_for_brax_training,
    )

    print(f"time to jit: {times[1] - times[0]}")
    print(f"time to train: {times[-1] - times[1]}")
else:
    # Load the latest brax checkpoint automatically
    from brax.training.agents.ppo import checkpoint as ppo_checkpoint
    from brax.training.agents.ppo import networks as ppo_networks
    from brax.training import checkpoint as brax_checkpoint, networks as brax_networks
    from pathlib import Path
    import json as _json

    checkpoint_dir = Path(f'./checkpoints/run_{CHECKPOINT_IDX}').resolve()
    # Find the latest checkpoint (highest step number)
    ckpt_dirs = sorted(
        [d for d in checkpoint_dir.iterdir() if d.is_dir() and d.name.isdigit()],
        key=lambda d: int(d.name)
    )
    latest_ckpt = ckpt_dirs[-1]
    print(f"Loading checkpoint: {latest_ckpt.name} (step {int(latest_ckpt.name):,})")

    # Load config manually, working around brax bug with null kernel init fns
    config_path = latest_ckpt / 'ppo_network_config.json'
    loaded_dict = _json.loads(config_path.read_text())
    nf_kwargs = loaded_dict['network_factory_kwargs']
    if 'activation' in nf_kwargs:
        nf_kwargs['activation'] = brax_networks.ACTIVATION[nf_kwargs['activation']]
    for k in ('policy_network_kernel_init_fn', 'value_network_kernel_init_fn',
              'q_network_kernel_init_fn', 'mean_kernel_init_fn'):
        if k in nf_kwargs:
            if nf_kwargs[k] is None:
                del nf_kwargs[k]
            else:
                nf_kwargs[k] = brax_networks.KERNEL_INITIALIZER[nf_kwargs[k]]
    nf_kwargs = {k: v for k, v in nf_kwargs.items() if v is not None}

    # Fix observation_size: convert {"state": {"shape": [52], ...}} -> {"state": 52, ...}
    obs_size_raw = loaded_dict['observation_size']
    if isinstance(obs_size_raw, dict):
        obs_size = {}
        for key, val in obs_size_raw.items():
            if isinstance(val, dict) and 'shape' in val:
                shape = val['shape']
                obs_size[key] = shape[0] if len(shape) == 1 else tuple(shape)
            else:
                obs_size[key] = val
    else:
        obs_size = obs_size_raw

    normalize_observations = loaded_dict.get('normalize_observations', True)
    action_size = loaded_dict['action_size']

    # Load params
    params = ppo_checkpoint.load(str(latest_ckpt))

    # Rebuild network from config — use plain dicts, not ConfigDict
    ppo_network = ppo_networks.make_ppo_networks(
        observation_size=obs_size,
        action_size=action_size,
        preprocess_observations_fn=(
            brax_checkpoint.running_statistics.normalize
            if normalize_observations
            else lambda x, y: x
        ),
        **nf_kwargs,
    )
    make_inference_fn = ppo_networks.make_inference_fn(ppo_network)
    inference_fn = make_inference_fn(params, deterministic=True)
    jit_inference_fn = jax.jit(inference_fn)
    print("Policy loaded successfully.")

## Visualize Rollouts

In [22]:
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
if TRAINING:
    jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

In [ ]:
!apt install -y ffmpeg

In [44]:
rng = jax.random.PRNGKey(42)
rollout = []

init_joint_position = [ 1.224399,   -1.10737256,  2.2151802,  -3.11259448, -1.40254952, -0.30685145,
 -1.10513002, -1.32389425,  2.09514085, -2.80322322, -1.78368266,  0.39448189]

# init_joint_position = [1.5, -1.8, 1.75, -1.25, -1.6, 0, -1.5, -1.8, 1.75, -1.25, -1.6, 0]


n_episodes = 1
env.num_batch = 2000
env.maxiter_cem = 4
cost_weights = jp.array([20, 20, 0.2, 1.5, 0.04, 5.0, 5.0, 0.5, 5, 5, 5, 0])

# Set episode length on the actual env config (not the separate env_cfg copy)
env._config.episode_length = 100

# Set home joint position (reset() reads self.home_joint_position)
env.home_joint_position = np.array(init_joint_position)
env.init_joint_position = np.array(init_joint_position)

# Re-create JIT wrappers so they pick up the new self.* attributes
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

In [ ]:
# env_cfg = registry.get_default_config(env_name)
# print(env_cfg.episode_length)
# print(env._config.episode_length)

print(env.home_joint_position)
print(env.init_joint_position)
# state = jit_reset(rng)
# print(state.data.qpos[env._joint_mask_pos])
# print(state)

[ 1.224 -1.107  2.215 -3.113 -1.403 -0.307 -1.105 -1.324  2.095 -2.803 -1.784  0.394]
[ 1.224 -1.107  2.215 -3.113 -1.403 -0.307 -1.105 -1.324  2.095 -2.803 -1.784  0.394]
[ 1.229 -1.106  2.206 -3.116 -1.393 -0.315 -1.103 -1.331  2.095 -2.812 -1.789  0.393]


In [ ]:

   
for _ in range(n_episodes):
  state = jit_reset(rng)

  for i in range(env_cfg.episode_length):
    act_rng, rng = jax.random.split(rng)
    obs_i = state.obs
    ctrl, _ = jit_inference_fn(obs_i, act_rng)
    state = jit_step(state, ctrl)
    rollout.append(state)

render_every = 1
frames = env.render(rollout[::render_every])
rewards = [s.reward for s in rollout]
media.show_video(frames, fps=1.0 / env.dt / render_every)

[ 1.491 -1.797  1.753 -1.252 -1.606  0.009 -1.497 -1.81   1.75  -1.246 -1.599  0.009]


NameError: name 'jit_inference_fn' is not defined

In [46]:

for _ in range(n_episodes):
  state = jit_reset(rng)
  print(state.data.qpos[env._joint_mask_pos])
#   rollout.append(state)
  rollout.append(state)
  for i in range(env._config.episode_length):
    act_rng, rng = jax.random.split(rng)
  
    state = jit_step(state, cost_weights)
    # state = jit_step(state, ctrl)
    rollout.append(state)


[ 1.229 -1.106  2.206 -3.116 -1.393 -0.315 -1.103 -1.331  2.095 -2.812 -1.789  0.393]


In [47]:
state

State(data=Data(time=Array(20., dtype=float64), qpos=Array([    0.641,     0.448,    -0.584,     0.498,     0.36 ,    -0.953,    -0.946,     0.312,
           0.218,    -0.23 ,     0.418,    -0.717,    -4.982,    -0.763, -1599.745,     1.   ,
           0.   ,    -0.   ,     0.004], dtype=float64), qvel=Array([   0.04 ,   -0.009,    0.014,   -0.013,    0.07 ,   -0.02 ,    0.048,   -0.023,    0.097,
          0.059,   -0.021,    0.066,   -0.167,   -0.027, -144.27 ,   -0.   ,    0.001,    0.   ],      dtype=float64), act=Array([], shape=(0,), dtype=float64), history=Array([], shape=(0,), dtype=float64), qacc_warmstart=Array([-0.   , -0.001,  0.002, -0.002,  0.001,  0.   ,  0.002,  0.002, -0.003, -0.007,  0.005,
        0.   ,  0.005,  0.001, -5.151,  0.   , -0.   , -0.   ], dtype=float64), plugin_state=Array([], shape=(0,), dtype=float64), ctrl=Array([], shape=(0,), dtype=float32), qfrc_applied=Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float64)

In [48]:
render_every =1
renderer = mujoco.Renderer(env._mj_model, height=480, width=640)
opt = mujoco.MjvOption()
# opt.flags[mujoco.mjtVisFlag.mjVIS_TRANSPARENT] = True
opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.lookat[:] = [0, 0, 0.4]
cam.distance = 2.0
cam.azimuth = 90
cam.elevation = -30

frames = []
for state in rollout[::render_every]:
    d = mujoco.MjData(env._mj_model)
    d.qpos, d.qvel = state.data.qpos, state.data.qvel
    d.mocap_pos, d.mocap_quat = state.data.mocap_pos, state.data.mocap_quat
    mujoco.mj_forward(env._mj_model, d)
    renderer.update_scene(d, camera=cam, scene_option=opt)

    # Draw planned EEF traces as spheres
    scn = renderer.scene
    eef_0_plan = np.array(state.info['eef_0_planned'])  # (num_steps, 7)
    eef_1_plan = np.array(state.info['eef_1_planned'])  # (num_steps, 7)

    for pts, color in [(eef_0_plan, [0, 0, 1, 0.8]),    # blue
                        (eef_1_plan, [0, 0, 1, 0.8])]:  
        for j in range(len(pts)):
            if scn.ngeom >= scn.maxgeom:
                break
            # Sphere at each planned waypoint
            g = scn.geoms[scn.ngeom]
            mujoco.mjv_initGeom(
                g,
                type=mujoco.mjtGeom.mjGEOM_SPHERE,
                size=[0.008, 0, 0],
                pos=pts[j, :3],
                mat=np.eye(3).flatten(),
                rgba=np.array(color, dtype=np.float32),
            )
            scn.ngeom += 1

            # Thin capsule connecting to next waypoint
            if j < len(pts) - 1 and scn.ngeom < scn.maxgeom:
                p0, p1 = pts[j, :3], pts[j+1, :3]
                mid = (p0 + p1) / 2.0
                diff = p1 - p0
                length = np.linalg.norm(diff)
                if length > 1e-6:
                    # Build rotation matrix: z-axis along diff
                    z = diff / length
                    # Pick a non-parallel vector for cross product
                    up = np.array([0, 0, 1]) if abs(z[2]) < 0.9 else np.array([1, 0, 0])
                    x = np.cross(up, z); x /= np.linalg.norm(x)
                    y = np.cross(z, x)
                    mat = np.column_stack([x, y, z]).flatten()

                    g = scn.geoms[scn.ngeom]
                    mujoco.mjv_initGeom(
                        g,
                        type=mujoco.mjtGeom.mjGEOM_CAPSULE,
                        size=[0.005, length / 2, 0],  # radius, half-length
                        pos=mid,
                        mat=mat,
                        rgba=np.array(color, dtype=np.float32),
                    )
                    scn.ngeom += 1

    frames.append(renderer.render())
renderer.close()

media.show_video(frames, fps=1.0 / env.dt / render_every)

This browser does not support the video tag.


In [ ]:
# --- State-level fields ---
rewards = jp.stack([s.reward for s in rollout])          # (T,)
dones = jp.stack([s.done for s in rollout])              # (T,)
obs_state = jp.stack([s.obs['state'] for s in rollout])             # (T, obs_dim)
obs_priviliged_state = jp.stack([s.obs['privileged_state'] for s in rollout])             # (T, obs_priv_dim)

# --- MJX Data fields (physics state) ---
qpos = jp.stack([s.data.qpos for s in rollout])          # (T, nq) - joint positions
qvel = jp.stack([s.data.qvel for s in rollout])          # (T, nv) - joint velocities
ctrl = jp.stack([s.data.ctrl for s in rollout])           # (T, nu) - applied controls
xpos = jp.stack([s.data.xpos for s in rollout])          # (T, nbody, 3) - body positions
xquat = jp.stack([s.data.xquat for s in rollout])        # (T, nbody, 4) - body orientations
site_xpos = jp.stack([s.data.site_xpos for s in rollout])  # (T, nsite, 3) - site positions
mocap_pos = jp.stack([s.data.mocap_pos for s in rollout])   # (T, nmocap, 3) - mocap positions
mocap_quat = jp.stack([s.data.mocap_quat for s in rollout]) # (T, nmocap, 4) - mocap orientations
qfrc_actuator = jp.stack([s.data.qfrc_actuator for s in rollout])  # (T, nv) - actuator forces
sensordata = jp.stack([s.data.sensordata for s in rollout])  # (T, nsensor) - sensor readings
xfrc_applied = jp.stack([s.data.xfrc_applied for s in rollout])  # (T, nbody, 6) - applied forces


# --- Info dict (env-specific) ---
steps = jp.stack([s.info['_steps'] for s in rollout])    # (T,)
target_1_pos = jp.stack([s.info['target_1_pos'] for s in rollout])  # (T, 3) - target_1 position
target_2_pos = jp.stack([s.info['target_2_pos'] for s in rollout])  # (T, 3) - target_2 position


print(f"Rollout length: {len(rollout)}")
print(f"qpos: {qpos.shape}, qvel: {qvel.shape}, ctrl: {ctrl.shape}")
print(f"xpos: {xpos.shape}, xquat: {xquat.shape}")
print(f"site_xpos: {site_xpos.shape}")
print(f"sensordata: {sensordata.shape}")
print(f"mocap_pos: {mocap_pos.shape}")
print(f"mocap_quat: {mocap_quat.shape}")
print(f"rewards: {rewards.shape}, obs: {obs_state.shape}, obs_priv: {obs_priviliged_state.shape}")
print(f"target_1_pos: {target_1_pos.shape}, target_2_pos: {target_2_pos.shape}")

In [ ]:
print("target_1_pos", target_1_pos)
print("target_2_pos", target_2_pos)

In [ ]:
plt.plot(rewards)
plt.show()

In [ ]:

plt.plot(qvel[:,env._joint_mask_vel])
plt.show()

In [ ]:
# print(obs_state[:,37:39])

In [ ]:
print(obs_priviliged_state.shape)
# task = take(1, "task")
# success = take(1, "success")
# cost_g_ball = take(1, "cost_g_ball")
# cost_r_ball = take(1, "cost_r_ball")
# current_cost_g = take(1, "current_cost_g")
# current_cost_r = take(1, "current_cost_r")
# penalty_collision = take(1, "penalty_collision_real_time")
state_len = obs_state.shape[1]
print(state_len)
# obs_priv = obs_priviliged_state[obs_state.shape[1]]
task = obs_priviliged_state[:, state_len + 0]
success = obs_priviliged_state[:, state_len + 1]
cost_g_ball = obs_priviliged_state[:, state_len + 2]
cost_r_ball = obs_priviliged_state[:, state_len + 3]
current_cost_g = obs_priviliged_state[:, state_len + 4]
current_cost_r = obs_priviliged_state[:, state_len + 5]
penalty_collision = obs_priviliged_state[:, state_len + 6]

print("task:", task)
print("success:", success)
print("cost_g_ball:", cost_g_ball)
print("cost_r_ball:", cost_r_ball)
print("current_cost_g:", current_cost_g)
print("current_cost_r:", current_cost_r)
print("penalty_collision:", penalty_collision)

In [ ]:
output_path = f"policy_output_{env_name}_{CHECKPOINT_IDX}.npz"
np.savez(
    output_path,
    policy_obs=np.asarray(jp.stack(policy_obs_list)),
    policy_actions=np.asarray(jp.stack(policy_actions)),
)
print(f"Saved {len(policy_actions)} steps to {output_path}")
d = np.load(output_path)
print(f"  policy_obs: {d['policy_obs'].shape}, policy_actions: {d['policy_actions'].shape}")

In [ ]:
d = np.load(f"policy_output_{env_name}_{CHECKPOINT_IDX}.npz")
act = d['policy_actions']  # (T, num_cost_weights) - cost weights from policy
obs = d['policy_obs']      # (T, obs_dim)
print(f"policy_actions: {act.shape}, policy_obs: {obs.shape}")

# Cost weight names from LiftBox.__init__ (env.cost_weights gets overwritten by JAX during step)
cost_names = [
    'collision_pick', 'collision_move', 'theta', 'velocity', 'z_axis',
    'distance_pick', 'distance_move', 'orientation', 'eef_to_obj',
    'obj_to_targ', 'object_orientation', 'smoothness',
]

fig, axes = plt.subplots(act.shape[1], 1, figsize=(14, 2.5 * act.shape[1]), sharex=True)
for i in range(act.shape[1]):
    # axes[i].plot(act[:, i]*env._base_costs[i])  # scale by base cost to get actual cost value
    # axes[i].plot(act[:, i]*env._base_scales)  # scale by base cost to get actual cost value
    axes[i].plot(jax.nn.softplus(act[:, i]) * env._base_scales[i])

    label = cost_names[i] 
    axes[i].set_ylabel(label, fontsize=14)
axes[0].set_title("Policy Output (Cost Weights)")
axes[-1].set_xlabel("timestep")
plt.tight_layout()
plt.show()

While the above policy is very simple, the work was extended using the Madrona batch renderer, and policies were transferred on a real robot. We encourage folks to check out the Madrona-MJX tutorial notebooks ([part 1](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/training_vision_1.ipynb) and [part 2](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/training_vision_2.ipynb))!

🙌 Thanks for stopping by The Playground!